# Faza 4 — srpski LipNet, transfer checkpoint-a i CTC backward


## Goal

Instanciraj 29-klasni srpski LipNet, prenesi svaki kompatibilan VIPL backbone/BiGRU
parametar, dokaži da su preskočeni samo `FC.weight` i `FC.bias`, pa uradi jedan
forward + CTC loss + backward na malom realnom batch-u. Nema trening epohe.


## Setup

Izaberi T4 GPU. Ovaj notebook namerno ne pokreće optimizer niti fine-tuning;
to počinje tek u Fazi 5.


In [ ]:
import subprocess, sys
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'editdistance>=0.8.1', 'opencv-python-headless>=4.10'],
    check=True,
)


In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/nikolabakic/Vizuelno-prepoznavanje-govora-na-osnovu-pokreta-usana-pomo-u-LipNet-modela.git'
REPO = Path('/content/lipnet-serbian')
if not (REPO / 'lipnet').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
os.chdir(REPO)
print('Repo:', REPO)
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
import numpy as np
import torch
assert torch.cuda.is_available(), 'Uključi T4 GPU u Colab Runtime postavkama.'
DEVICE = torch.device('cuda')
torch.manual_seed(0)
torch.cuda.manual_seed_all(0)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
from google.colab import drive

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/LipNet')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print('Drive izlaz:', DRIVE_ROOT)


## Steps

### 1. Učitaj Phase 2 frejmove i anotacije


In [ ]:
import zipfile

MOUTH_ARCHIVE = DRIVE_ROOT / 'ai_speak_lip.zip'
SOURCE_ARCHIVE = Path('/content/drive/MyDrive/processed.zip')  # promeni po potrebi
MOUTH_ROOT = Path('/content/ai_speak_lip')
ALIGN_EXTRACT = Path('/content/ai_speak_align')
assert MOUTH_ARCHIVE.exists() and SOURCE_ARCHIVE.exists()
if not next(MOUTH_ROOT.glob('spk*/video/video_a/*'), None):
    MOUTH_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(MOUTH_ARCHIVE) as archive:
        archive.extractall(MOUTH_ROOT)
if not next(ALIGN_EXTRACT.rglob('spk*/alignment/*.align'), None):
    ALIGN_EXTRACT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(SOURCE_ARCHIVE) as archive:
        members = [member for member in archive.infolist()
                   if '/alignment/' in f'/{member.filename}' and member.filename.endswith('.align')]
        archive.extractall(ALIGN_EXTRACT, members=members)
CORPUS_ROOT = next(ALIGN_EXTRACT.rglob('spk*/alignment/*.align')).parents[2]


### 2. Preuzmi pinovani engleski checkpoint i audituj transfer


In [ ]:
from urllib.request import urlretrieve

from lipnet.dataset import SERBIAN_LETTERS
from lipnet.model import LipNet
from lipnet.train import load_vipl_transfer

UPSTREAM_SHA = '40209e09c49553c00c25c7d41faa3706aea3c625'
CHECKPOINT = Path('/content') / 'LipNet_unseen_loss_0.44562849402427673_wer_0.1332580699113564_cer_0.06796452465503355.pt'
CHECKPOINT_URL = (
    'https://raw.githubusercontent.com/VIPL-Audio-Visual-Speech-Understanding/'
    f'LipNet-PyTorch/{UPSTREAM_SHA}/pretrain/LipNet_unseen_loss_0.44562849402427673_wer_0.1332580699113564_cer_0.06796452465503355.pt'
)
if not CHECKPOINT.exists():
    urlretrieve(CHECKPOINT_URL, CHECKPOINT)

NUM_CLASSES = 1 + len(SERBIAN_LETTERS)
assert NUM_CLASSES == 29
model = LipNet(num_classes=NUM_CLASSES).to(DEVICE)
audit = load_vipl_transfer(model, CHECKPOINT)
assert set(audit.skipped_shape) == {'FC.weight', 'FC.bias'}
assert not audit.missing_in_checkpoint and not audit.unexpected_in_checkpoint
assert all(name in audit.loaded for name in ('conv1.weight', 'conv2.weight', 'conv3.weight'))
assert any(name.startswith('gru1.') for name in audit.loaded)
assert any(name.startswith('gru2.') for name in audit.loaded)
print(audit.summary())


### 3. Izaberi dva kratka realna klipa koja zadovoljavaju CTC i napravi batch


In [ ]:
from data.splits import TRAIN_SPEAKERS
from lipnet.dataset import SerbianDataset, minimum_ctc_steps, variable_length_collate

dataset = SerbianDataset(MOUTH_ROOT, CORPUS_ROOT, TRAIN_SPEAKERS, phase='train')
ordered_indices = sorted(
    range(len(dataset)),
    key=lambda index: len(list(dataset.data[index][0].glob('*.jpg'))),
)
samples = []
for index in ordered_indices:
    sample = dataset[index]
    target = sample['txt'][:sample['txt_len']]
    if minimum_ctc_steps(target) <= sample['vid_len']:
        samples.append(sample)
    if len(samples) == 2:
        break
assert len(samples) == 2, 'Nisu pronađena dva CTC-validna klipa.'
batch = variable_length_collate(samples)
print('Batch:', tuple(batch['vid'].shape))
print('vid_len:', batch['vid_len'].tolist(), 'txt_len:', batch['txt_len'].tolist())


## Checks

### 4. Jedan GPU forward, konačan CTC loss i backward audit


In [ ]:
from lipnet.train import backward_smoke_step

loss, logits_shape = backward_smoke_step(model, batch, DEVICE)
assert logits_shape[0] == 2 and logits_shape[-1] == NUM_CLASSES
assert np.isfinite(loss)
print('Logits (B,T,C):', logits_shape)
print('CTC loss:', loss)
print('PASS: svi trainable parametri imaju konačne gradijente.')


In [ ]:
import json

result = {
    'phase': 4,
    'upstream_commit': UPSTREAM_SHA,
    'num_classes': NUM_CLASSES,
    'loaded_tensors': len(audit.loaded),
    'skipped_shape': list(audit.skipped_shape),
    'missing_in_checkpoint': list(audit.missing_in_checkpoint),
    'unexpected_in_checkpoint': list(audit.unexpected_in_checkpoint),
    'batch_shape': list(batch['vid'].shape),
    'logits_shape': list(logits_shape),
    'ctc_loss': loss,
    'backward': 'passed',
    'torch_version': torch.__version__,
    'gpu': torch.cuda.get_device_name(0),
}
result_path = DRIVE_ROOT / 'phase4_transfer_ctc_audit.json'
result_path.write_text(json.dumps(result, indent=2) + '\n', encoding='utf-8')
print(json.dumps(result, indent=2))


## Next Steps

Faza 4 je završena samo ako audit preskače tačno dva FC tenzora, CTC loss je
konačan i svi gradijenti su konačni. Tek tada Faza 5 sme da uvede optimizer,
checkpoint-e i baseline fine-tuning.
